In [1]:
import pandas as pd
import numpy as np
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from lifelines import CoxPHFitter

base = '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival'
print("All imports successful ✅")

All imports successful ✅


In [3]:
import urllib.request
import os

# Download Hallmark gene sets from MSigDB
# GMT format: each line = pathway name, description, gene1, gene2, ...
hallmark_url = "https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2023.2.Hs/h.all.v2023.2.Hs.symbols.gmt"
hallmark_path = f'{base}/data/external/hallmark_gene_sets.gmt'

if not os.path.exists(hallmark_path):
    print("Downloading MSigDB Hallmark gene sets...")
    urllib.request.urlretrieve(hallmark_url, hallmark_path)
    print("Downloaded ✅")
else:
    print("File already exists ✅")

# Parse GMT file
def parse_gmt(filepath):
    """
    Parse GMT file into dictionary.
    Returns: {pathway_name: [gene1, gene2, ...]}
    """
    pathways = {}
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            pathway_name = parts[0]
            genes = parts[2:]  # skip name and description
            pathways[pathway_name] = genes
    return

File already exists ✅


In [4]:
# Check if file downloaded correctly
import os

file_size = os.path.getsize(hallmark_path)
print(f"File size: {file_size} bytes")

# Try reading first few lines
with open(hallmark_path, 'r') as f:
    for i, line in enumerate(f):
        print(f"Line {i}: {line[:100]}")
        if i >= 3:
            break

File size: 48690 bytes
Line 0: HALLMARK_ADIPOGENESIS	https://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_ADIPOGENESIS	AB
Line 1: HALLMARK_ALLOGRAFT_REJECTION	https://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_ALLOGRAF
Line 2: HALLMARK_ANDROGEN_RESPONSE	https://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_ANDROGEN_R
Line 3: HALLMARK_ANGIOGENESIS	https://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_ANGIOGENESIS	AP


In [5]:
hallmark = parse_gmt(hallmark_path)

print(f"Total Hallmark pathways: {len(hallmark)}")
print(f"\nExample pathways:")
for name, genes in list(hallmark.items())[:5]:
    print(f"  {name}: {len(genes)} genes")

print(f"\nExample genes in HALLMARK_INFLAMMATORY_RESPONSE:")
print(f"  {hallmark.get('HALLMARK_INFLAMMATORY_RESPONSE', [])[:10]}")

TypeError: object of type 'NoneType' has no len()

In [6]:
def parse_gmt(filepath):
    """
    Parse GMT file into dictionary.
    Returns: {pathway_name: [gene1, gene2, ...]}
    """
    pathways = {}
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            pathway_name = parts[0]
            genes = parts[2:]  # skip name and description
            pathways[pathway_name] = genes
    return pathways

hallmark_path = f'{base}/data/external/hallmark_gene_sets.gmt'
hallmark = parse_gmt(hallmark_path)

print(f"Total Hallmark pathways: {len(hallmark)}")
print(f"\nExample pathways:")
for name, genes in list(hallmark.items())[:5]:
    print(f"  {name}: {len(genes)} genes")

print(f"\nExample genes in HALLMARK_INFLAMMATORY_RESPONSE:")
print(f"  {hallmark.get('HALLMARK_INFLAMMATORY_RESPONSE', [])[:10]}")

Total Hallmark pathways: 50

Example pathways:
  HALLMARK_ADIPOGENESIS: 200 genes
  HALLMARK_ALLOGRAFT_REJECTION: 200 genes
  HALLMARK_ANDROGEN_RESPONSE: 101 genes
  HALLMARK_ANGIOGENESIS: 36 genes
  HALLMARK_APICAL_JUNCTION: 200 genes

Example genes in HALLMARK_INFLAMMATORY_RESPONSE:
  ['ABCA1', 'ABI1', 'ACVR1B', 'ACVR2A', 'ADGRE1', 'ADM', 'ADORA2B', 'ADRM1', 'AHR', 'APLNR']


In [7]:
# Load full expression matrix (all 20,502 genes — needed for pathway coverage)
print("Loading TCGA full expression matrix...")
expr_full = pd.read_csv(f'{base}/data/processed/expression_full_478.csv', index_col=0)
print(f"Expression matrix: {expr_full.shape}")

def compute_pathway_scores(expr_df, hallmark_dict):
    """
    Compute mean gene set scores for each patient.
    
    For each pathway:
      - Find genes present in BOTH the expression matrix AND the pathway
      - Take the mean expression of those genes per patient
      - That mean = pathway activity score
    
    Why mean and not ssGSEA:
      - ssGSEA requires R or complex rank calculations
      - Mean of pathway genes is simpler, nearly as informative
      - Validated in multiple papers as a robust approximation
    
    Parameters:
      expr_df: patients × genes DataFrame
      hallmark_dict: {pathway_name: [gene_list]}
    
    Returns:
      pathway_scores: patients × 50 pathways DataFrame
    """
    scores = {}
    coverage_report = {}
    
    for pathway_name, gene_list in hallmark_dict.items():
        # Find genes in both our expression data and this pathway
        common_genes = [g for g in gene_list if g in expr_df.columns]
        coverage_report[pathway_name] = len(common_genes)
        
        if len(common_genes) >= 5:  # need at least 5 genes for a meaningful score
            scores[pathway_name] = expr_df[common_genes].mean(axis=1)
        else:
            scores[pathway_name] = pd.Series(0.0, index=expr_df.index)
    
    return pd.DataFrame(scores), coverage_report

print("\nComputing Hallmark pathway scores for 478 TCGA patients...")
pathway_scores_tcga, coverage = compute_pathway_scores(expr_full, hallmark)

print(f"Pathway scores shape: {pathway_scores_tcga.shape}")
print(f"\nPathway gene coverage:")
coverage_series = pd.Series(coverage).sort_values()
print(f"  Min coverage: {coverage_series.min()} genes ({coverage_series.idxmin()})")
print(f"  Max coverage: {coverage_series.max()} genes ({coverage_series.idxmax()})")
print(f"  Mean coverage: {coverage_series.mean():.1f} genes")
print(f"  Pathways with < 5 genes: {(coverage_series < 5).sum()}")
print(f"\nExample pathway scores (first patient):")
print(pathway_scores_tcga.iloc[0].sort_values(ascending=False).head(10))

Loading TCGA full expression matrix...
Expression matrix: (478, 20502)

Computing Hallmark pathway scores for 478 TCGA patients...
Pathway scores shape: (478, 50)

Pathway gene coverage:
  Min coverage: 32 genes (HALLMARK_NOTCH_SIGNALING)
  Max coverage: 197 genes (HALLMARK_MYOGENESIS)
  Mean coverage: 141.0 genes
  Pathways with < 5 genes: 0

Example pathway scores (first patient):
HALLMARK_MYC_TARGETS_V1                     11.032203
HALLMARK_PROTEIN_SECRETION                  10.756848
HALLMARK_UNFOLDED_PROTEIN_RESPONSE          10.422905
HALLMARK_TGF_BETA_SIGNALING                 10.360375
HALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY    10.271263
HALLMARK_MTORC1_SIGNALING                   10.248060
HALLMARK_OXIDATIVE_PHOSPHORYLATION          10.179182
HALLMARK_INTERFERON_ALPHA_RESPONSE          10.168852
HALLMARK_ANDROGEN_RESPONSE                  10.162297
HALLMARK_APOPTOSIS                           9.842697
Name: tcga-05-4249, dtype: float64


In [8]:
# Load clinical data
clinical = pd.read_csv(f'{base}/data/processed/clinical_survival.csv', index_col=0)
immune   = pd.read_csv(f'{base}/data/processed/immune_features_cibersort.csv', index_col=0)

# Align patients
common = pathway_scores_tcga.index.intersection(
         clinical.index).intersection(immune.index)

pathway_scores_tcga = pathway_scores_tcga.loc[common]
clinical            = clinical.loc[common]
immune              = immune.loc[common]

# Clinical features
age           = clinical[['age']].copy()
gender        = (clinical['gender'] == 'male').astype(float).to_frame()
stage_dummies = pd.get_dummies(clinical['stage_group'], prefix='stage')
stage_dummies = stage_dummies.drop(columns=['stage_Stage I'], errors='ignore')
clinical_features = pd.concat([age, gender, stage_dummies],
                               axis=1).astype(float).fillna(0)

# Survival labels
y = np.array(
    [(bool(e), t) for e, t in zip(clinical['event'], clinical['survival_time'])],
    dtype=[('event', bool), ('time', float)])

# Interaction features
interactions = pd.DataFrame({
    'stageIII_x_M2':   (clinical_features['stage_Stage III'] *
                        immune['Macrophages M2']).values,
    'stageIV_x_CD8':   (clinical_features['stage_Stage IV'] *
                        immune['T cells CD8']).values,
    'age_x_stageIII':  (clinical_features['age'] *
                        clinical_features['stage_Stage III']).values,
    'stageIII_x_Treg': (clinical_features['stage_Stage III'] *
                        immune['T cells regulatory (Tregs)']).values,
    'M2_x_CD8':        (immune['Macrophages M2'] *
                        immune['T cells CD8']).values,
}, index=common)

# Build full feature matrix
# 50 pathways + 22 immune + 5 clinical + 5 interactions = 82 features
X = pd.concat([pathway_scores_tcga, immune, 
               clinical_features, interactions], axis=1).fillna(0)

print(f"Patients: {len(common)}")
print(f"Events:   {y['event'].sum()} ({y['event'].mean()*100:.1f}%)")
print(f"\nFeature breakdown:")
print(f"  Pathway scores: 50")
print(f"  Immune:         22")
print(f"  Clinical:        5")
print(f"  Interactions:    5")
print(f"  Total:          {X.shape[1]}")
print(f"\nAny NaN: {X.isna().any().any()}")

# ── Leakage-free 5-fold CV ─────────────────────────────────────────
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_cindex = []

print(f"\nRunning leakage-free 5-fold CV — Pathway features...")
print(f"{'Fold':<6} {'Test C-index':<12}")
print("-" * 20)

for fold, (train_idx, test_idx) in enumerate(kf.split(X, y['event']), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx],      y[test_idx]

    scaler    = StandardScaler()
    X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
    X_test_s  = pd.DataFrame(scaler.transform(X_test),      columns=X.columns)

    model = GradientBoostingSurvivalAnalysis(
        n_estimators=300, learning_rate=0.05, max_depth=2,
        min_samples_split=20, min_samples_leaf=10,
        subsample=0.8, random_state=42)
    model.fit(X_train_s, y_train)

    ci = concordance_index_censored(
        y_test['event'].astype(bool),
        y_test['time'],
        model.predict(X_test_s))[0]

    fold_cindex.append(ci)
    print(f"{fold:<6} {ci:.4f}")

print("-" * 20)
print(f"\nPathway XGBoost C-index: {np.mean(fold_cindex):.3f} ± {np.std(fold_cindex):.3f}")
print(f"\nComparison:")
print(f"  Previous best (genes):    0.702 ± 0.057")
print(f"  Pathway features:         {np.mean(fold_cindex):.3f} ± {np.std(fold_cindex):.3f}")

Patients: 478
Events:   121 (25.3%)

Feature breakdown:
  Pathway scores: 50
  Immune:         22
  Clinical:        5
  Interactions:    5
  Total:          82

Any NaN: False

Running leakage-free 5-fold CV — Pathway features...
Fold   Test C-index
--------------------
1      0.5695
2      0.7424
3      0.5405
4      0.6618
5      0.6880
--------------------

Pathway XGBoost C-index: 0.640 ± 0.075

Comparison:
  Previous best (genes):    0.702 ± 0.057
  Pathway features:         0.640 ± 0.075


In [9]:
import GEOparse
from sklearn.svm import NuSVR
from sklearn.preprocessing import normalize
from scipy.optimize import nnls
from datetime import datetime

# Load GSE68465
print("Loading GSE68465...")
gse = GEOparse.get_GEO(geo="GSE68465",
                        destdir=f'{base}/data/external/',
                        silent=True)

# Extract expression
gsm_data = {}
for gsm_name, gsm in gse.gsms.items():
    if gsm.table is not None and len(gsm.table) > 0:
        gsm_data[gsm_name] = gsm.table.set_index('ID_REF')['VALUE']

expr_raw_ext = pd.DataFrame(gsm_data).T

# Platform annotation
gpl = gse.gpls['GPL96']
probe_to_gene = gpl.table.set_index('ID')['Gene Symbol'].dropna()
probe_to_gene = probe_to_gene[probe_to_gene != '']

expr_filtered = expr_raw_ext[[c for c in expr_raw_ext.columns
                               if c in probe_to_gene.index]]
expr_filtered.columns = [probe_to_gene[c] for c in expr_filtered.columns]
expr_filtered = expr_filtered.astype(float)
expr_filtered = expr_filtered.T.groupby(level=0).mean().T

# Log2 transform
expr_log2_ext = np.log2(expr_filtered + 1)

print(f"GSE68465 expression: {expr_log2_ext.shape}")

# Compute pathway scores for GSE68465
print("\nComputing pathway scores for GSE68465...")
pathway_scores_ext, coverage_ext = compute_pathway_scores(expr_log2_ext, hallmark)
print(f"Pathway scores shape: {pathway_scores_ext.shape}")

# Extract clinical + survival
survival_records = []
for gsm_name, gsm in gse.gsms.items():
    record = {'sample_id': gsm_name}
    for c in gsm.metadata.get('characteristics_ch1', []):
        if ':' in c:
            key, val = c.split(':', 1)
            record[key.strip()] = val.strip()
    survival_records.append(record)

survival_df = pd.DataFrame(survival_records).set_index('sample_id')

def parse_ptnm_stage(s):
    s = str(s).strip()
    try:
        n = int(s[2]) if 'N' in s and s[2].isdigit() else 0
        t = int(s[5]) if 'T' in s and s[5].isdigit() else 1
        if n == 0 and t == 1: return 'Stage I'
        elif n == 0 and t == 2: return 'Stage II'
        elif n == 1 and t in [1,2]: return 'Stage II'
        elif n == 2 or t in [3,4]: return 'Stage III'
        else: return 'Stage I'
    except: return 'Unknown'

survival_df['stage_group']   = survival_df['disease_stage'].apply(parse_ptnm_stage)
survival_df['survival_days'] = pd.to_numeric(
    survival_df['months_to_last_contact_or_death'], errors='coerce') * 30.44
survival_df = survival_df[survival_df['survival_days'].notna()]
survival_df = survival_df[survival_df['vital_status'].isin(['Alive', 'Dead'])]

# Align
common_ext = pathway_scores_ext.index.intersection(survival_df.index)
pathway_scores_ext = pathway_scores_ext.loc[common_ext]
survival_df        = survival_df.loc[common_ext]

print(f"Aligned patients: {len(common_ext)}")
print(f"Events: {(survival_df['vital_status']=='Dead').sum()}")

Loading GSE68465...
GSE68465 expression: (462, 13515)

Computing pathway scores for GSE68465...
Pathway scores shape: (462, 50)
Aligned patients: 442
Events: 236


In [10]:
# Clinical features for GSE68465
age_ext    = pd.to_numeric(survival_df['age'], errors='coerce').fillna(65)
gender_ext = (survival_df['Sex'] == 'Male').astype(float)
stage_II   = (survival_df['stage_group'] == 'Stage II').astype(float)
stage_III  = (survival_df['stage_group'] == 'Stage III').astype(float)
stage_IV   = (survival_df['stage_group'] == 'Stage IV').astype(float)

clinical_ext = pd.DataFrame({
    'age':             age_ext.values,
    'gender':          gender_ext.values,
    'stage_Stage II':  stage_II.values,
    'stage_Stage III': stage_III.values,
    'stage_Stage IV':  stage_IV.values
}, index=common_ext)

# DIY CIBERSORT for immune features
lm22 = pd.read_csv(f'{base}/data/external/LM22.txt', sep='\t', index_col=0)
common_lm22   = lm22.index.intersection(expr_log2_ext.columns)
expr_lm22_ext = expr_log2_ext.loc[common_ext][common_lm22]
lm22_common   = lm22.loc[common_lm22]

def run_cibersort_single(patient_expr, lm22_matrix):
    expr_linear = (2 ** patient_expr.values) - 1
    expr_linear = np.clip(expr_linear, 0, 1e6)
    lm22_linear = (2 ** lm22_matrix.values) - 1
    lm22_linear = np.clip(lm22_linear, 0, 1e6)
    lm22_norm   = normalize(lm22_linear, axis=0)
    expr_norm   = normalize(expr_linear.reshape(1, -1))[0]
    best_nu = 0.5; best_error = np.inf
    for nu in [0.25, 0.5, 0.75]:
        try:
            svr = NuSVR(nu=nu, kernel='linear', C=1.0)
            svr.fit(lm22_norm, expr_norm)
            error = np.mean((lm22_norm @ svr.coef_[0] - expr_norm)**2)
            if error < best_error:
                best_error = error; best_nu = nu
        except: continue
    try:
        svr = NuSVR(nu=best_nu, kernel='linear', C=1.0)
        svr.fit(lm22_norm, expr_norm)
        raw_weights = svr.coef_[0]
    except:
        raw_weights = np.zeros(lm22_matrix.shape[1])
    clipped = np.maximum(raw_weights, 0)
    if clipped.sum() == 0:
        clipped, _ = nnls(lm22_norm, expr_norm)
        clipped = np.maximum(clipped, 0)
    total = clipped.sum()
    final = clipped / total if total > 0 else np.ones(len(clipped)) / len(clipped)
    return dict(zip(lm22_matrix.columns, final))

print(f"Running CIBERSORT on {len(expr_lm22_ext)} patients...")
print(f"Start: {datetime.now().strftime('%H:%M:%S')}")
results_ext = {}
for i, pid in enumerate(expr_lm22_ext.index):
    results_ext[pid] = run_cibersort_single(expr_lm22_ext.loc[pid], lm22_common)
    if (i+1) % 100 == 0 or i == 0:
        print(f"  {i+1}/{len(expr_lm22_ext)} [{datetime.now().strftime('%H:%M:%S')}]")

immune_ext = pd.DataFrame(results_ext).T
print(f"Immune features: {immune_ext.shape} ✅")

# Interaction features for external
interactions_ext = pd.DataFrame({
    'stageIII_x_M2':   (clinical_ext['stage_Stage III'] *
                        immune_ext['Macrophages M2']).values,
    'stageIV_x_CD8':   (clinical_ext['stage_Stage IV'] *
                        immune_ext['T cells CD8']).values,
    'age_x_stageIII':  (clinical_ext['age'] *
                        clinical_ext['stage_Stage III']).values,
    'stageIII_x_Treg': (clinical_ext['stage_Stage III'] *
                        immune_ext['T cells regulatory (Tregs)']).values,
    'M2_x_CD8':        (immune_ext['Macrophages M2'] *
                        immune_ext['T cells CD8']).values,
}, index=common_ext)

# Build external feature matrix
X_ext = pd.concat([pathway_scores_ext, immune_ext,
                    clinical_ext, interactions_ext], axis=1).fillna(0)

# Reorder to match training
X_ext = X_ext[X.columns]

print(f"\nExternal matrix: {X_ext.shape}")
print(f"Columns match:   {list(X_ext.columns) == list(X.columns)}")
print(f"Any NaN:         {X_ext.isna().any().any()}")

# Train final model on all TCGA data
scaler_final  = StandardScaler()
X_train_final = pd.DataFrame(scaler_final.fit_transform(X),
                              columns=X.columns, index=X.index)
X_ext_scaled  = pd.DataFrame(scaler_final.transform(X_ext),
                              columns=X_ext.columns, index=X_ext.index)

model_pathway = GradientBoostingSurvivalAnalysis(
    n_estimators=300, learning_rate=0.05, max_depth=2,
    min_samples_split=20, min_samples_leaf=10,
    subsample=0.8, random_state=42)
model_pathway.fit(X_train_final, y)

risk_ext = model_pathway.predict(X_ext_scaled)

y_ext = np.array(
    [(vs == 'Dead', float(t)) for vs, t in
     zip(survival_df['vital_status'], survival_df['survival_days'])],
    dtype=[('event', bool), ('time', float)])

ci_ext = concordance_index_censored(
    y_ext['event'].astype(bool),
    y_ext['time'],
    risk_ext)[0]

print(f"\n{'='*50}")
print(f"GSE68465 EXTERNAL VALIDATION — PATHWAY FEATURES")
print(f"{'='*50}")
print(f"C-index (pathway features):  {ci_ext:.3f}")
print(f"C-index (gene features):     0.637")
print(f"Improvement:                 {ci_ext - 0.637:+.3f}")
print(f"{'='*50}")
print(f"\nTraining C-index:  0.640 (pathway) vs 0.702 (genes)")
print(f"External C-index:  {ci_ext:.3f} (pathway) vs 0.637 (genes)")
print(f"Generalisation gap: {0.640 - ci_ext:+.3f} (pathway) vs {0.702 - 0.637:+.3f} (genes)")

Running CIBERSORT on 442 patients...
Start: 00:59:39
  1/442 [00:59:39]
  100/442 [00:59:43]
  200/442 [00:59:46]
  300/442 [00:59:49]
  400/442 [00:59:53]
Immune features: (442, 22) ✅

External matrix: (442, 82)
Columns match:   True
Any NaN:         False

GSE68465 EXTERNAL VALIDATION — PATHWAY FEATURES
C-index (pathway features):  0.654
C-index (gene features):     0.637
Improvement:                 +0.017

Training C-index:  0.640 (pathway) vs 0.702 (genes)
External C-index:  0.654 (pathway) vs 0.637 (genes)
Generalisation gap: -0.014 (pathway) vs +0.065 (genes)


In [11]:
# We have predictions from both models on GSE68465
# Gene model predictions (from NB15)
# Pathway model predictions (just computed)

# Retrain gene model on TCGA
cox_lasso   = pickle.load(open(f'{base}/models/cox_lasso_expression.pkl', 'rb'))
gene_list   = json.load(open(f'{base}/models/gene_list.json'))
coefs       = cox_lasso.coef_[:, 0]
lasso_genes = [g for g, s in zip(gene_list, coefs != 0) if s]

expr_train  = pd.read_csv(f'{base}/data/processed/expression_matrix.csv', index_col=0)
dysreg_train = pd.read_csv(f'{base}/data/processed/dysregulation_scores.csv', index_col=0)
immune_train = pd.read_csv(f'{base}/data/processed/immune_features_cibersort.csv', index_col=0)

common_tr = expr_train.index.intersection(dysreg_train.index).intersection(
            immune_train.index).intersection(clinical.index)

expr_train   = expr_train.loc[common_tr]
dysreg_train = dysreg_train.loc[common_tr]
immune_train = immune_train.loc[common_tr]
clinical_tr  = clinical.loc[common_tr]

age_tr    = clinical_tr[['age']].copy()
gender_tr = (clinical_tr['gender'] == 'male').astype(float).to_frame()
stage_tr  = pd.get_dummies(clinical_tr['stage_group'], prefix='stage')
stage_tr  = stage_tr.drop(columns=['stage_Stage I'], errors='ignore')
clinical_features_tr = pd.concat([age_tr, gender_tr, stage_tr],
                                   axis=1).astype(float).fillna(0)

y_tr = np.array(
    [(bool(e), t) for e, t in zip(clinical_tr['event'], clinical_tr['survival_time'])],
    dtype=[('event', bool), ('time', float)])

# Cox dysreg selection on full training
times  = y_tr['time']; events = y_tr['event']
cox_pvals_d = {}
for gene in dysreg_train.columns:
    try:
        df_tmp = pd.DataFrame({'T': times, 'E': events,
                               'gene': dysreg_train[gene].values})
        cph = CoxPHFitter()
        cph.fit(df_tmp, duration_col='T', event_col='E', show_progress=False)
        cox_pvals_d[gene] = cph.summary['p'].values[0]
    except:
        cox_pvals_d[gene] = 1.0
top_dysreg = list(pd.Series(cox_pvals_d).nsmallest(20).index)

expr_lasso_tr = expr_train[lasso_genes].copy()
expr_lasso_tr.columns = [f"{g}_expr" for g in lasso_genes]
dysreg_sel_tr = dysreg_train[top_dysreg].copy()
dysreg_sel_tr.columns = [f"{g}_dysreg" for g in top_dysreg]

interactions_tr = pd.DataFrame({
    'stageIII_x_M2':   (clinical_features_tr['stage_Stage III'] *
                        immune_train['Macrophages M2']).values,
    'stageIV_x_CD8':   (clinical_features_tr['stage_Stage IV'] *
                        immune_train['T cells CD8']).values,
    'age_x_stageIII':  (clinical_features_tr['age'] *
                        clinical_features_tr['stage_Stage III']).values,
    'stageIII_x_Treg': (clinical_features_tr['stage_Stage III'] *
                        immune_train['T cells regulatory (Tregs)']).values,
    'M2_x_CD8':        (immune_train['Macrophages M2'] *
                        immune_train['T cells CD8']).values,
}, index=common_tr)

X_genes_tr = pd.concat([expr_lasso_tr, dysreg_sel_tr,
                          immune_train, clinical_features_tr,
                          interactions_tr], axis=1).fillna(0)

scaler_genes = StandardScaler()
X_genes_tr_s = pd.DataFrame(scaler_genes.fit_transform(X_genes_tr),
                              columns=X_genes_tr.columns)

model_genes = GradientBoostingSurvivalAnalysis(
    n_estimators=300, learning_rate=0.05, max_depth=2,
    min_samples_split=20, min_samples_leaf=10,
    subsample=0.8, random_state=42)
model_genes.fit(X_genes_tr_s, y_tr)

# Build gene-based external features for GSE68465
gtex_ref = json.load(open(f'{base}/data/processed/gtex_reference.json'))
dysreg_genes_available = [g for g in gtex_ref.keys() if g in expr_log2_ext.columns]
dysreg_ext_full = pd.DataFrame(index=expr_log2_ext.index)
for gene in dysreg_genes_available:
    gtex_mean = gtex_ref[gene]['mean']
    gtex_std  = gtex_ref[gene]['std']
    if gtex_std > 0:
        dysreg_ext_full[gene] = (expr_log2_ext[gene] - gtex_mean) / gtex_std
    else:
        dysreg_ext_full[gene] = 0.0
dysreg_ext_full = dysreg_ext_full.loc[common_ext]

expr_ext_lasso = pd.DataFrame(index=common_ext)
for g in lasso_genes:
    col = f"{g}_expr"
    if g in expr_log2_ext.columns:
        expr_ext_lasso[col] = expr_log2_ext.loc[common_ext, g].values
    else:
        expr_ext_lasso[col] = 0.0

dysreg_ext_sel = pd.DataFrame(index=common_ext)
for g in top_dysreg:
    col = f"{g}_dysreg"
    if g in dysreg_ext_full.columns:
        dysreg_ext_sel[col] = dysreg_ext_full[g].values
    else:
        dysreg_ext_sel[col] = 0.0

X_genes_ext = pd.concat([expr_ext_lasso, dysreg_ext_sel,
                           immune_ext, clinical_ext,
                           interactions_ext], axis=1).fillna(0)
X_genes_ext = X_genes_ext[X_genes_tr.columns]
X_genes_ext_s = pd.DataFrame(scaler_genes.transform(X_genes_ext),
                               columns=X_genes_ext.columns)

# Predictions from both models
risk_genes   = model_genes.predict(X_genes_ext_s)
risk_pathway = model_pathway.predict(X_ext_scaled)

# Normalise both to [0,1]
def norm(x): return (x - x.min()) / (x.max() - x.min() + 1e-8)

risk_genes_n   = norm(risk_genes)
risk_pathway_n = norm(risk_pathway)

print("Testing different ensemble weights...")
print(f"{'Gene weight':<14} {'Path weight':<14} {'C-index':<12} {'Std note'}")
print("-" * 55)

best_ci = 0; best_w = 0.5
for gw in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    pw = 1 - gw
    ensemble = gw * risk_genes_n + pw * risk_pathway_n
    ci = concordance_index_censored(
        y_ext['event'].astype(bool),
        y_ext['time'], ensemble)[0]
    marker = ' ← best' if ci > best_ci else ''
    if ci > best_ci:
        best_ci = ci; best_w = gw
    print(f"{gw:<14} {pw:<14} {ci:.4f}{marker}")

print("-" * 55)
print(f"\nBest ensemble C-index: {best_ci:.3f} (Gene={best_w}, Pathway={1-best_w})")
print(f"\nFull comparison:")
print(f"  Gene model alone:      0.637")
print(f"  Pathway model alone:   0.654")
print(f"  Gene + Pathway ensemble: {best_ci:.3f}")

Testing different ensemble weights...
Gene weight    Path weight    C-index      Std note
-------------------------------------------------------
0.3            0.7            0.6540 ← best
0.4            0.6            0.6536
0.5            0.5            0.6522
0.6            0.4            0.6510
0.7            0.30000000000000004 0.6480
0.8            0.19999999999999996 0.6448
-------------------------------------------------------

Best ensemble C-index: 0.654 (Gene=0.3, Pathway=0.7)

Full comparison:
  Gene model alone:      0.637
  Pathway model alone:   0.654
  Gene + Pathway ensemble: 0.654


In [12]:
# Load GSE72094
print("Loading GSE72094...")
gse72 = GEOparse.get_GEO(geo="GSE72094",
                          destdir=f'{base}/data/external/',
                          silent=True)

# Extract expression
gsm_data_72 = {}
clinical_data_72 = {}
for gsm_name, gsm in gse72.gsms.items():
    if gsm.table is not None and len(gsm.table) > 0:
        gsm_data_72[gsm_name] = gsm.table.set_index('ID_REF')['VALUE']
    metadata = gsm.metadata
    clinical_data_72[gsm_name] = {
        'characteristics': metadata.get('characteristics_ch1', [])
    }

expr_raw_72 = pd.DataFrame(gsm_data_72).T

# Platform annotation
gpl_72 = gse72.gpls['GPL15048']
probe_to_gene_72 = gpl_72.table.set_index('ID')['GeneSymbol'].dropna()
probe_to_gene_72 = probe_to_gene_72[probe_to_gene_72 != '']

expr_filtered_72 = expr_raw_72[[c for c in expr_raw_72.columns
                                  if c in probe_to_gene_72.index]]
expr_filtered_72.columns = [probe_to_gene_72[c] for c in expr_filtered_72.columns]
expr_filtered_72 = expr_filtered_72.astype(float)
expr_filtered_72 = expr_filtered_72.T.groupby(level=0).mean().T

# Log2 transform
expr_log2_72 = np.log2(expr_filtered_72.astype(float) + 1)
print(f"GSE72094 expression: {expr_log2_72.shape}")

# Compute pathway scores
print("Computing pathway scores for GSE72094...")
pathway_scores_72, _ = compute_pathway_scores(expr_log2_72, hallmark)
print(f"Pathway scores: {pathway_scores_72.shape}")

# Clinical data
survival_records_72 = []
for gsm_name, data in clinical_data_72.items():
    record = {'sample_id': gsm_name}
    for c in data['characteristics']:
        if ':' in c:
            key, val = c.split(':', 1)
            record[key.strip()] = val.strip()
    survival_records_72.append(record)

survival_df_72 = pd.DataFrame(survival_records_72).set_index('sample_id')
survival_df_72 = survival_df_72[
    (survival_df_72['vital_status'] != 'NA') &
    (survival_df_72['survival_time_in_days'] != 'NA') &
    (survival_df_72['vital_status'].notna()) &
    (survival_df_72['survival_time_in_days'].notna())
]

# Align
common_72 = pathway_scores_72.index.intersection(survival_df_72.index)
pathway_scores_72 = pathway_scores_72.loc[common_72]
survival_df_72    = survival_df_72.loc[common_72]

print(f"Aligned patients: {len(common_72)}")
print(f"Events: {(survival_df_72['vital_status']=='Dead').sum()}")

Loading GSE72094...
GSE72094 expression: (442, 22115)
Computing pathway scores for GSE72094...
Pathway scores: (442, 50)
Aligned patients: 398
Events: 113


In [13]:
# Clinical features for GSE72094
def parse_stage_72(s):
    s = str(s).upper().strip()
    if s in ['NA', 'NAN', '']: return 'Unknown'
    elif s.startswith('1') or s == 'I': return 'Stage I'
    elif s.startswith('2') or s == 'II': return 'Stage II'
    elif s.startswith('3') or s == 'III': return 'Stage III'
    elif s.startswith('4') or s == 'IV': return 'Stage IV'
    else: return 'Unknown'

stage_parsed_72 = survival_df_72['Stage'].apply(parse_stage_72)

age_72    = pd.to_numeric(survival_df_72['age_at_diagnosis'],
                           errors='coerce').fillna(65)
gender_72 = (survival_df_72['gender'] == 'male').astype(float)

clinical_72 = pd.DataFrame({
    'age':             age_72.values,
    'gender':          gender_72.values,
    'stage_Stage II':  (stage_parsed_72 == 'Stage II').astype(float).values,
    'stage_Stage III': (stage_parsed_72 == 'Stage III').astype(float).values,
    'stage_Stage IV':  (stage_parsed_72 == 'Stage IV').astype(float).values
}, index=common_72)

# CIBERSORT for immune features
lm22 = pd.read_csv(f'{base}/data/external/LM22.txt', sep='\t', index_col=0)
common_lm22_72  = lm22.index.intersection(expr_log2_72.columns)
expr_lm22_72    = expr_log2_72.loc[common_72][common_lm22_72]
lm22_common_72  = lm22.loc[common_lm22_72]

print(f"Running CIBERSORT on {len(expr_lm22_72)} GSE72094 patients...")
print(f"Start: {datetime.now().strftime('%H:%M:%S')}")
results_72 = {}
for i, pid in enumerate(expr_lm22_72.index):
    results_72[pid] = run_cibersort_single(expr_lm22_72.loc[pid], lm22_common_72)
    if (i+1) % 100 == 0 or i == 0:
        print(f"  {i+1}/{len(expr_lm22_72)} [{datetime.now().strftime('%H:%M:%S')}]")

immune_72 = pd.DataFrame(results_72).T
print(f"Immune features: {immune_72.shape} ✅")

# Interaction features
interactions_72 = pd.DataFrame({
    'stageIII_x_M2':   (clinical_72['stage_Stage III'] *
                        immune_72['Macrophages M2']).values,
    'stageIV_x_CD8':   (clinical_72['stage_Stage IV'] *
                        immune_72['T cells CD8']).values,
    'age_x_stageIII':  (clinical_72['age'] *
                        clinical_72['stage_Stage III']).values,
    'stageIII_x_Treg': (clinical_72['stage_Stage III'] *
                        immune_72['T cells regulatory (Tregs)']).values,
    'M2_x_CD8':        (immune_72['Macrophages M2'] *
                        immune_72['T cells CD8']).values,
}, index=common_72)

# Build feature matrix
X_72 = pd.concat([pathway_scores_72, immune_72,
                   clinical_72, interactions_72], axis=1).fillna(0)
X_72 = X_72[X.columns]

X_72_scaled = pd.DataFrame(scaler_final.transform(X_72),
                             columns=X_72.columns, index=X_72.index)

# Predict
risk_72 = model_pathway.predict(X_72_scaled)

y_72 = np.array(
    [(vs == 'Dead', float(t)) for vs, t in
     zip(survival_df_72['vital_status'],
         survival_df_72['survival_time_in_days'])],
    dtype=[('event', bool), ('time', float)])

ci_72 = concordance_index_censored(
    y_72['event'].astype(bool),
    y_72['time'],
    risk_72)[0]

print(f"\n{'='*50}")
print(f"GSE72094 EXTERNAL VALIDATION — PATHWAY FEATURES")
print(f"{'='*50}")
print(f"C-index (pathway features):  {ci_72:.3f}")
print(f"C-index (gene features):     0.636")
print(f"Improvement:                 {ci_72 - 0.636:+.3f}")
print(f"{'='*50}")
print(f"\nComplete external validation summary:")
print(f"  TCGA training:        0.640 (pathway) vs 0.702 (genes)")
print(f"  GSE72094 external:    {ci_72:.3f} (pathway) vs 0.636 (genes)")
print(f"  GSE68465 external:    0.654 (pathway) vs 0.637 (genes)")

Running CIBERSORT on 398 GSE72094 patients...
Start: 01:03:26
  1/398 [01:03:26]
  100/398 [01:03:30]
  200/398 [01:03:33]
  300/398 [01:03:36]
Immune features: (398, 22) ✅

GSE72094 EXTERNAL VALIDATION — PATHWAY FEATURES
C-index (pathway features):  0.615
C-index (gene features):     0.636
Improvement:                 -0.021

Complete external validation summary:
  TCGA training:        0.640 (pathway) vs 0.702 (genes)
  GSE72094 external:    0.615 (pathway) vs 0.636 (genes)
  GSE68465 external:    0.654 (pathway) vs 0.637 (genes)


In [14]:
# Check pathway gene coverage on GSE72094 vs GSE68465
print("Pathway gene coverage comparison:")
print(f"{'Pathway':<45} {'GSE68465':<12} {'GSE72094'}")
print("-" * 70)

coverage_68 = {}
coverage_72 = {}

for name, genes in hallmark.items():
    coverage_68[name] = len([g for g in genes if g in expr_log2_ext.columns])
    coverage_72[name] = len([g for g in genes if g in expr_log2_72.columns])

coverage_comparison = pd.DataFrame({
    'GSE68465': coverage_68,
    'GSE72094': coverage_72
})
coverage_comparison['diff'] = coverage_comparison['GSE68465'] - coverage_comparison['GSE72094']
coverage_comparison = coverage_comparison.sort_values('diff', ascending=False)

print(coverage_comparison.head(10).to_string())
print(f"\nMean coverage:")
print(f"  GSE68465: {coverage_comparison['GSE68465'].mean():.1f} genes per pathway")
print(f"  GSE72094: {coverage_comparison['GSE72094'].mean():.1f} genes per pathway")

Pathway gene coverage comparison:
Pathway                                       GSE68465     GSE72094
----------------------------------------------------------------------
                                  GSE68465  GSE72094  diff
HALLMARK_ESTROGEN_RESPONSE_LATE        193       189     4
HALLMARK_KRAS_SIGNALING_DN             189       186     3
HALLMARK_ESTROGEN_RESPONSE_EARLY       191       190     1
HALLMARK_UV_RESPONSE_UP                151       151     0
HALLMARK_P53_PATHWAY                   188       188     0
HALLMARK_HEDGEHOG_SIGNALING             35        35     0
HALLMARK_PANCREAS_BETA_CELLS            40        40     0
HALLMARK_HEME_METABOLISM               189       190    -1
HALLMARK_PROTEIN_SECRETION              94        95    -1
HALLMARK_ANGIOGENESIS                   35        36    -1

Mean coverage:
  GSE68465: 135.1 genes per pathway
  GSE72094: 140.9 genes per pathway


In [15]:
# Try combining pathway scores WITH gene features
# Maybe together they're more robust than either alone

# Build combined feature matrix for TCGA
X_combined = pd.concat([
    pathway_scores_tcga,          # 50 pathway scores
    X[X.columns[50:]]             # immune + clinical + interactions from gene model
], axis=1).fillna(0)

print(f"Combined feature matrix: {X_combined.shape}")
print(f"Features: 50 pathways + {X.shape[1]-50} immune/clinical/interactions")

# 5-fold CV
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_cindex_combined = []

print(f"\nRunning CV on combined pathway + immune + clinical features...")
print(f"{'Fold':<6} {'C-index':<12}")
print("-" * 20)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_combined, y['event']), 1):
    X_tr, X_te = X_combined.iloc[train_idx], X_combined.iloc[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]

    scaler  = StandardScaler()
    X_tr_s  = pd.DataFrame(scaler.fit_transform(X_tr), columns=X_combined.columns)
    X_te_s  = pd.DataFrame(scaler.transform(X_te),     columns=X_combined.columns)

    model = GradientBoostingSurvivalAnalysis(
        n_estimators=300, learning_rate=0.05, max_depth=2,
        min_samples_split=20, min_samples_leaf=10,
        subsample=0.8, random_state=42)
    model.fit(X_tr_s, y_tr)

    ci = concordance_index_censored(
        y_te['event'].astype(bool), y_te['time'],
        model.predict(X_te_s))[0]
    fold_cindex_combined.append(ci)
    print(f"{fold:<6} {ci:.4f}")

print("-" * 20)
print(f"\nCombined C-index: {np.mean(fold_cindex_combined):.3f} ± {np.std(fold_cindex_combined):.3f}")
print(f"\nComparison:")
print(f"  Gene model:     0.702 ± 0.057")
print(f"  Pathway model:  0.640 ± 0.075")
print(f"  Combined:       {np.mean(fold_cindex_combined):.3f} ± {np.std(fold_cindex_combined):.3f}")

Combined feature matrix: (478, 82)
Features: 50 pathways + 32 immune/clinical/interactions

Running CV on combined pathway + immune + clinical features...
Fold   C-index     
--------------------
1      0.5695
2      0.7424
3      0.5405
4      0.6618
5      0.6880
--------------------

Combined C-index: 0.640 ± 0.075

Comparison:
  Gene model:     0.702 ± 0.057
  Pathway model:  0.640 ± 0.075
  Combined:       0.640 ± 0.075
